In [ ]:
!pip -q install textstat pandas

In [ ]:
import re
import textstat
import pandas as pd

# Starter list only. Ask the team to review/expand this later.
MEDICAL_TERMS = {
    "diagnosis", "prognosis", "symptom", "symptoms", "treatment",
    "medication", "medications", "therapy", "therapeutic",
    "clinical", "chronic", "acute", "inflammation", "infection",
    "syndrome", "disease", "disorder", "anatomy", "physiology",
    "biopsy", "screening", "contraindication", "adverse",
    "intravenous", "antibiotic", "anticoagulant", "hypertension",
    "diabetes", "cardiovascular", "neurological"
}

def evaluate_readability(text, max_grade=8, max_words=None):
    """Return shared readability metrics for one patient explanation."""
    if not isinstance(text, str) or not text.strip():
        return {
            "status": "error",
            "error": "Text is empty or invalid."
        }

    clean_text = " ".join(text.split())
    words = re.findall(r"\b[a-zA-Z'-]+\b", clean_text.lower())
    word_count = len(words)
    sentence_count = max(len(re.findall(r"[.!?]+", clean_text)), 1)

    jargon_words = sorted({word for word in words if word in MEDICAL_TERMS})
    jargon_count = sum(word in MEDICAL_TERMS for word in words)
    jargon_density = jargon_count / word_count if word_count else 0

    fk_grade = textstat.flesch_kincaid_grade(clean_text)

    # SMOG is less reliable for very short explanations.
    smog_grade = textstat.smog_index(clean_text) if sentence_count >= 3 else None

    grade_pass = fk_grade <= max_grade
    length_pass = True if max_words is None else word_count <= max_words

    return {
        "status": "ok",
        "word_count": word_count,
        "sentence_count": sentence_count,
        "flesch_kincaid_grade": round(fk_grade, 2),
        "smog_grade": round(smog_grade, 2) if smog_grade is not None else None,
        "medical_jargon_count": jargon_count,
        "medical_jargon_density": round(jargon_density, 4),
        "medical_terms_found": ", ".join(jargon_words),
        "passes_grade_target": grade_pass,
        "passes_length_target": length_pass,
        "passes_readability": grade_pass and length_pass,
    }

In [ ]:
sample_text = """
High blood pressure means the force of blood pushing against your blood vessel walls is too high.
It may not cause symptoms, but treatment can lower the risk of heart disease and stroke.
"""

evaluate_readability(sample_text)

{'status': 'ok',
 'word_count': 33,
 'sentence_count': 2,
 'flesch_kincaid_grade': 5.51,
 'smog_grade': None,
 'medical_jargon_count': 3,
 'medical_jargon_density': 0.0909,
 'medical_terms_found': 'disease, symptoms, treatment',
 'passes_grade_target': True,
 'passes_length_target': True,
 'passes_readability': True}

In [ ]:
hard_text = """
Hypertension is a chronic cardiovascular condition characterized by persistently elevated arterial pressure.
Pharmacological intervention may be clinically indicated to prevent cardiovascular complications.
"""

evaluate_readability(hard_text)

{'status': 'ok',
 'word_count': 22,
 'sentence_count': 2,
 'flesch_kincaid_grade': 25.71,
 'smog_grade': None,
 'medical_jargon_count': 4,
 'medical_jargon_density': 0.1818,
 'medical_terms_found': 'cardiovascular, chronic, hypertension',
 'passes_grade_target': False,
 'passes_length_target': True,
 'passes_readability': False}

In [ ]:
three_sentence_text = """
High blood pressure means the force of blood against your blood vessel walls is too high.
It can raise your risk of heart disease and stroke.
Medicine, exercise, and healthy food can help lower it.
"""

evaluate_readability(three_sentence_text)

{'status': 'ok',
 'word_count': 35,
 'sentence_count': 3,
 'flesch_kincaid_grade': 4.13,
 'smog_grade': 7.79,
 'medical_jargon_count': 1,
 'medical_jargon_density': 0.0286,
 'medical_terms_found': 'disease',
 'passes_grade_target': True,
 'passes_length_target': True,
 'passes_readability': True}

In [ ]:
test_examples = pd.DataFrame({
    "question_id": ["easy_1", "hard_1"],
    "model_output": [
        sample_text,
        hard_text
    ]
})

results = pd.concat(
    [test_examples,
     test_examples["model_output"].apply(evaluate_readability).apply(pd.Series)],
    axis=1
)

results

,question_id,model_output,status,word_count,sentence_count,flesch_kincaid_grade,smog_grade,medical_jargon_count,medical_jargon_density,medical_terms_found,passes_grade_target,passes_length_target,passes_readability
0,easy_1,\nHigh blood pressure means the force of blood...,ok,33,2,5.51,None,3,0.0909,"disease, symptoms, treatment",True,True,True
1,hard_1,\nHypertension is a chronic cardiovascular con...,ok,22,2,25.71,None,4,0.1818,"cardiovascular, chronic, hypertension",False,True,False


In [ ]:
results.to_csv("test_readability_results.csv", index=False)

from google.colab import files
files.download("test_readability_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>